# Inference Varroa: Faster R-CNN và YOLOv8s

Chạy `best.pt` đã train trên ảnh mới hoặc thư mục ảnh. Dùng pipeline RGB/letterbox,
FP32 và ngưỡng confidence đã chọn trên validation giống lượt đánh giá Kaggle.

Cài dependency bằng `python -m pip install -r requirements.txt` trong môi trường
của kernel nếu chưa có; xem README.md. Notebook không tải weights hay train lại.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "labs/Lab-3_ObjectDetection"]
LAB = next(p.resolve() for p in candidates if (p / "inference/predict.py").is_file())
sys.path.insert(0, str(LAB / "inference"))

SOURCE = LAB / "dataset/varroa_ready/images/test"  # Đổi thành ảnh/thư mục của bạn.
OUTPUT = LAB / "inference/results"
MODEL = "both"  # both, fasterrcnn, yolov8s
DEVICE = "auto"  # auto, cpu, cuda:0
LIMIT = 8  # None: chạy tất cả ảnh.
CONFIDENCE = None  # None: ngưỡng validation riêng từng model.
THREADS = 4


In [ ]:
import torch
from predict import DEFAULT_WEIGHTS, run_inference

torch.set_num_threads(THREADS)
# Có thể đổi đường dẫn trong WEIGHTS khi dùng checkpoint khác.
WEIGHTS = dict(DEFAULT_WEIGHTS)
result = run_inference(SOURCE, OUTPUT, model=MODEL, device=DEVICE,
                       confidence=CONFIDENCE, limit=LIMIT, weights=WEIGHTS)


In [ ]:
import json

families = ["fasterrcnn", "yolov8s"] if MODEL == "both" else [MODEL]
for family in families:
    run = json.loads((result / family / "run.json").read_text())
    print(f"{family}: {run['image_count']} ảnh, {run['detection_count']} detections, "
          f"confidence={run['confidence']:.4f}, device={run['device']}")
    print("Predictions:", result / family / "predictions.json")


Box đỏ là dự đoán, nhãn gồm `varroa` và confidence. Khi chạy cả hai model,
Faster R-CNN ở bên trái và YOLOv8s ở bên phải. Ảnh không có detection vẫn được lưu.
JSON dùng box `[x, y, width, height]` theo pixel ảnh gốc; `run.json` ánh xạ ID sang ảnh.
Các ảnh dưới đây chỉ minh họa inference, không thay thế đánh giá trên toàn bộ test.


In [ ]:
from IPython.display import Image, display

preview = result / "comparison" if MODEL == "both" else result / MODEL / "images"
for path in sorted(preview.rglob("*.png"))[:4]:
    print(path.relative_to(result))
    display(Image(filename=str(path)))
